# Generalised Estimating Equations (GEE)

## Overview
GEE is a **marginal (population-average) approach** to correlated data. Rather than explicitly modelling the within-cluster correlation structure (as mixed models do), GEE treats the correlation as a nuisance and estimates it as a working correlation matrix, producing unbiased regression coefficients with robust (sandwich) standard errors.

**GEE vs Mixed Models:**

| Feature | GEE | Mixed Models (LMM/GLMM) |
|---|---|---|
| Parameter interpretation | Population-average | Subject-specific (conditional on random effects) |
| Random effects | No | Yes |
| Correlation structure | Working (approximate) | Modelled explicitly |
| Robustness to misspecified correlation | High (robust SE) | Lower |
| Missing data handling | Requires MAR | Better under MCAR/MAR |
| Primary use | Estimating population-level effects | Individual-level predictions, variance partitioning |

**When GEE is preferred:** when the research question is about population-average effects (e.g., what is the average effect of the treatment across all individuals?) and you do not need individual-level predictions or random effect estimates.

**Reference:** Quinn & Keough (2002) ch. 13; Zuur et al. (2009)

---

In [ ]:
library(tidyverse); library(geepack); library(lme4)
set.seed(73)
# Repeated measurements of disease severity on 30 plants over 5 time points
# Half treated with fungicide; measurements within plant are correlated
n_plants <- 30
n_times  <- 5
plant_id  <- rep(1:n_plants, each = n_times)
treatment <- rep(ifelse(1:n_plants <= 15, "control", "fungicide"), each = n_times)
time      <- rep(1:n_times, times = n_plants)

# True effect: fungicide reduces severity; severity increases over time
plant_re <- rep(rnorm(n_plants, 0, 0.8), each = n_times)  # individual plant effect
log_mu   <- 1.5 + 0.3 * time +
            ifelse(treatment == "fungicide", -0.6, 0) +
            plant_re
dat <- data.frame(
  plant_id  = factor(plant_id),
  treatment = factor(treatment),
  time      = time,
  severity  = rpois(n_plants * n_times, lambda = exp(log_mu))
)

cat("Dataset: n =", nrow(dat), "observations,",
    n_plants, "plants ×", n_times, "time points\n")
dat |> group_by(treatment, time) |>
  summarise(mean_sev = round(mean(severity), 1), .groups = "drop") |>
  pivot_wider(names_from = treatment, values_from = mean_sev) |> print()

---
## Fitting GEE with different working correlation structures

In [ ]:
# GEE requires data sorted by cluster (plant_id), then time within cluster
dat <- dat[order(dat$plant_id, dat$time), ]

# Independence working correlation (ignores clustering — baseline)
gee_ind <- geeglm(severity ~ treatment + time,
                  id = plant_id, data = dat,
                  family = poisson(link = "log"),
                  corstr = "independence")

# Exchangeable (compound symmetry): all within-cluster pairs equally correlated
gee_exch <- geeglm(severity ~ treatment + time,
                   id = plant_id, data = dat,
                   family = poisson(link = "log"),
                   corstr = "exchangeable")

# AR1: correlation decays with increasing lag (natural for time series)
gee_ar1 <- geeglm(severity ~ treatment + time,
                  id = plant_id, data = dat,
                  family = poisson(link = "log"),
                  corstr = "ar1")

# Unstructured: each pair has its own correlation
gee_unst <- geeglm(severity ~ treatment + time,
                   id = plant_id, data = dat,
                   family = poisson(link = "log"),
                   corstr = "unstructured")

cat("=== Estimated working correlation parameter (alpha) ===\n")
cat("Independence:   (no correlation estimated)\n")
cat(sprintf("Exchangeable:   α = %.3f\n", gee_exch$geese$alpha))
cat(sprintf("AR1:            α = %.3f\n", gee_ar1$geese$alpha))
cat("Unstructured:  (full matrix, see summary)\n")

In [ ]:
# Compare estimates and robust SE across correlation structures
cat("--- Coefficient estimates and robust SE ---\n")
for (model_name in c("gee_ind", "gee_exch", "gee_ar1")) {
  mod <- get(model_name)
  coef_tab <- coef(summary(mod))
  cat(sprintf("\n%s (%s):\n", model_name,
              mod$corstr))
  print(round(coef_tab[, c("Estimate", "Std.err", "Pr(>|W|)")], 4))
}
cat("\nNote: coefficients should be similar across working structures\n")
cat("(robustness of GEE); SE may differ — use the structure closest to truth\n")

In [ ]:
# Compare GEE (marginal) vs GLMM (subject-specific)
glmm_fit <- glmer(severity ~ treatment + time + (1 | plant_id),
                  data = dat, family = poisson(link = "log"),
                  control = glmerControl(optimizer = "bobyqa"))

cat("=== GEE (exchangeable) vs GLMM: treatment effect comparison ===\n")
cat(sprintf("GEE treatment coef:  %.3f (SE = %.3f) [population-average]\n",
            coef(summary(gee_exch))["treatmentfungicide", "Estimate"],
            coef(summary(gee_exch))["treatmentfungicide", "Std.err"]))
cat(sprintf("GLMM treatment coef: %.3f (SE = %.3f) [subject-specific]\n",
            fixef(glmm_fit)["treatmentfungicide"],
            sqrt(vcov(glmm_fit)["treatmentfungicide", "treatmentfungicide"])))

cat("\nWith a log link and random effects, the GLMM coefficient is larger in magnitude\n")
cat("(subject-specific effects are always further from zero than marginal effects\n")
cat(" for GLMs with non-identity links — this is expected, not an error)\n")

---
## Common Pitfalls

**1. Misunderstanding the marginal vs subject-specific distinction**
For linear models with identity links, GEE and LMM give identical fixed-effect estimates. For GLMs with log or logit links, they give different estimates that answer different questions. A GEE coefficient describes the average treatment effect across the population; a GLMM coefficient describes the effect for a specific individual (conditional on their random effect). Neither is wrong — they answer different questions.

**2. Choosing a working correlation structure without thought**
GEE estimates are valid with any working structure (due to robust SE), but efficiency is best when the working structure approximates the truth. Use exchangeable for clusters with no natural ordering (e.g., subsamples within a plot) and AR1 for time series within a subject. QIC (Quasi-Information Criterion, `QIC()` in geepack) can guide selection.

**3. Ignoring data ordering requirements**
GEE in R requires data sorted by cluster ID, then by the within-cluster ordering variable. Unsorted data produces wrong correlation estimates and potentially wrong coefficients. Always sort before fitting.

**4. Applying GEE with very small numbers of clusters**
The robust sandwich SE is asymptotically valid as the number of clusters → ∞. With fewer than 30 clusters, the sandwich estimator can be downward-biased (anticonservative). Use the small-sample correction (`std.err = "san.se"` is default; consider `"fij"` or `"j1"` for small k).

**5. Using GEE when you need individual-level predictions**
GEE has no random effects and produces no subject-specific predictions. If you need to predict outcomes for new individuals or partition variance between levels, use a mixed model.

**6. Treating GEE as identical to robust regression**
GEE uses within-cluster correlation as the working structure; robust (sandwich) regression in linear models uses heteroskedasticity-consistent errors. These address different sources of non-independence and are not interchangeable.


---
*r_methods_library - Samantha McGarrigle*